# Day 20 — Inflation & Cost of Living Dashboard: Analysis

Reproduces the headline numbers from the processed CSVs (never the raw FRED files) and rebuilds the 4 charts. Source: U.S. Bureau of Labor Statistics via FRED. Latest data: October 2026.

In [ ]:
from pathlib import Path

import pandas as pd


def _find_root() -> Path:
    """Project root, from the notebook file location (Day 1 lesson)."""
    here = Path.cwd()
    for parent in [here, *here.parents]:
        if (parent / "data" / "processed" / "monthly_panel.csv").exists():
            return parent
    raise FileNotFoundError("could not find data/processed/monthly_panel.csv")


ROOT = _find_root()
PROC = ROOT / "data" / "processed"
print("root:", ROOT)

## Load the processed tables

In [ ]:
panel = pd.read_csv(PROC / "monthly_panel.csv", parse_dates=["date"])
yoy_df = pd.read_csv(PROC / "yoy_inflation.csv", parse_dates=["date"])
cum = pd.read_csv(PROC / "category_cumulative.csv", parse_dates=["date"])
real = pd.read_csv(PROC / "real_earnings.csv", parse_dates=["date"])
basket = pd.read_csv(PROC / "basket_cost.csv", parse_dates=["date"])
summary = pd.read_csv(PROC / "summary_stats.csv")
print(panel.shape, "months:", panel["date"].iloc[0].date(), "->", panel["date"].iloc[-1].date())

## Headline inflation

In [ ]:
latest_yoy = yoy_df["headline_yoy_pct"].dropna().iloc[-1]
latest_yoy_date = yoy_df["date"][yoy_df["headline_yoy_pct"].notna()].iloc[-1]
peak = yoy_df["headline_yoy_pct"].max()
peak_date = yoy_df["date"][yoy_df["headline_yoy_pct"].idxmax()]
print(f"Latest YoY: {latest_yoy:.2f}% ({latest_yoy_date.date()})")
print(f"Peak YoY: {peak:.2f}% ({peak_date.date()})")
print("Oct 2026 YoY is undefined: BLS published no Oct 2025 CPI (2025 lapse in appropriations).")

## Category divergence since Mar 2006

In [ ]:
last = cum.iloc[-1]
for cat in ["food_at_home", "energy", "housing", "transportation", "apparel", "medical_care", "education"]:
    print(f"{cat:15s} +{last[f'{cat}_cum_pct']:6.1f}%  (vs headline {last[f'{cat}_divergence_pp']:+.1f} pp)")
print(f"Headline cumulative: +{last['headline_cum_pct']:.1f}%")

## Real pay vs. prices

In [ ]:
print(f"Nominal hourly pay: ${real['nominal_usd_per_hour'].iloc[0]:.2f} -> ${real['nominal_usd_per_hour'].iloc[-1]:.2f}")
print(f"Real hourly pay (1982-84 $): ${real['real_usd_per_hour'].iloc[0]:.2f} -> ${real['real_usd_per_hour'].iloc[-1]:.2f}")
print(f"Real pay indexed: {real['real_indexed'].iloc[-1]:.1f} (Mar 2006 = 100)")
print(f"Real YoY Sep 2026: {real['real_yoy_pct'].iloc[-2]:.2f}% (Oct 2026 undefined)")

## The $100 basket

In [ ]:
print(f"$100 basket (Mar 2006) costs ${basket['basket_cost_usd'].iloc[-1]:.2f} in Oct 2026")
print(f"Purchasing power of $1: ${basket['purchasing_power_usd'].iloc[-1]:.2f}")

## Rebuild the 4 charts

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.make_charts import build_all  # noqa: E402

for path in build_all():
    print(path, path.stat().st_size, "bytes")

## Takeaway

Prices are up 69% since Mar 2006 while real hourly pay is up only 12% — and in Sep 2026 real pay actually fell 0.5% year on year as energy surged 20%. The averages hide the split: housing, food and medical care are up ~80% each since 2006, apparel only 15%.